# 95. 支持向量机（SVM）

<!-- module-learning-arc:start -->
> **机器学习 模块主线｜第 10 / 34 步：扩展监督/无监督模型工具箱**
>
> **持续应用背景：** 建设可信预测系统：从统一训练流程开始，比较模型、处理不平衡、选择阈值、解释结果并保存完整 Pipeline，最终回答模型能否安全投入使用。
>
> **承接上一阶段：** 梯度提升模型  →  **本章任务：** 支持向量机（SVM）  →  **下一步：** 朴素贝叶斯
>
> **大作业连接：** 本章练习将成为《模型上线评审会》的一部分，最终需要把候选模型变成经过预测合同、泄漏审计、业务阈值、错误分析和模型卡检查的上线建议。
<!-- module-learning-arc:end -->


## 本章场景

**背景引入**：拿到一张带标签的样本表，你常常要先回答“哪些样本是一类”——这笔交易是否可疑、这位客户会不会流失。支持向量机（SVM）擅长在两类数据之间找到一条“间隔最大”的分界线，让新样本被清楚地分到两边；再配合不同的核函数，它既能处理简单的线性可分问题，也能画出弯曲边界去拟合更复杂的关系，是分类任务里很实用的一把尺子。



## 本章目标

学完本章，你将能够：

- **理解**：理解「支持向量机（SVM）」的核心思想、适用场景、关键假设与要解释的业务问题。
- **操作**：能按标准流程完成数据准备、模型训练与评估，并解读「支持向量机（SVM）」的关键输出指标。
- **迁移**：能把「支持向量机（SVM）」迁移到一份新数据上，独立完成任务并就结果给出有分寸的结论。


## 95.1 核心概念

**背景引入**：我们常想在两类点之间划一条“最宽的分界线”，让两类离得越远越稳。支持向量机（SVM）做的正是这件事：它找一条尽量宽的“安全护栏”来分开两类，而护栏放哪只由最贴边的那几个点（支持向量）说了算。理解 C 与 gamma，就知道它如何在“严格分类”和“容忍噪声”之间取舍。


- 支持向量决定分类间隔（打个比方：在两类中间拉一条最宽的“安全护栏”，护栏放哪只由最贴边的那几个点说了算——中间的多数点反而帮不上忙。）
- C 大时更强调训练误差、正则更弱
- RBF gamma 控制单个样本影响范围
- SVC 在超大样本上训练成本较高


## 95.2 方法分类速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 线性核与 RBF 核 | `rows.append()`、`m.score()`、`n_support_.sum()`、`pd.DataFrame()` | 同一标准化流水线下比较两种核函数。 | 不缩放就调 C 和 gamma |
| C 与 gamma | `grid.append()`、`m.score()`、`pd.DataFrame()`、`.fit()` | 参数共同控制 RBF 边界复杂度。 | 在大数据上直接运行核 SVM |


## 95.3 示例 1：线性核与 RBF 核

同一标准化流水线下比较两种核函数。


<!-- math-foundation:chapter-95 -->
### 数学推导｜SVM 最大化分类间隔

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜分类边界为** $w^Tx+b=0$，两个规范化支持边界为 $w^Tx+b=\pm1$。

**第 2 步｜两条支持边界之间的距离是**

$$
margin=\frac{2}{\lVert w\rVert}
$$

因此最大化间隔等价于最小化 $\lVert w\rVert^2/2$。

**第 3 步｜用松弛变量允许少量违例。** $\xi_i$ 衡量样本违反间隔的程度，目标加上 $C\sum_i\xi_i$。较大 $C$ 更强调少犯训练错误，较小 $C$ 更强调宽间隔。

**把上面的关系收束为本章计算式：**

$$
\min_{w,b}\frac{1}{2}\lVert w\rVert^2+C\sum_i\xi_i,\qquad y_i(w^Tx_i+b)\ge1-\xi_i
$$

**符号解释：** $C$ 控制间隔宽度与训练错误之间的权衡。

**代码对应：** 标准化特征，并在验证集比较 `C`、核函数和 `gamma`。

**使用边界：** 核 SVM 在大样本上成本高；决策分数不是天然概率。


In [ ]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, random_state=84
)
rows = []
for kernel in ["linear", "rbf"]:
    m = make_pipeline(StandardScaler(), SVC(kernel=kernel, C=1)).fit(
        X_train, y_train
    )
    rows.append(
        [
            kernel,
            m.score(X_train, y_train),
            m.score(X_test, y_test),
            m.named_steps["svc"].n_support_.sum(),
        ]
    )
display(pd.DataFrame(rows, columns=["kernel", "train", "test", "支持向量数"]))


**练一练**：在示例 1 的基础上，把比较的核函数从 `["linear", "rbf"]` 增加一个 `"poly"`，其余设置（StandardScaler 标准化、C=1、数据切分与随机种子）保持不变，重新训练并记录三种核函数的测试集分数。

提示：多数代码可以直接复用示例，你只需要改 `kernel_list` 这一行。


In [ ]:
# 请在下方填写代码
# 修改点：把核函数列表补全为 ["linear", "rbf", "poly"]
kernel_list = ["linear", "rbf"]  # TODO: 请把 "poly" 加进列表

rows2 = []
for kernel in kernel_list:
    m = make_pipeline(StandardScaler(), SVC(kernel=kernel, C=1)).fit(
        X_train, y_train
    )
    rows2.append([kernel, round(m.score(X_test, y_test), 3)])
result = pd.DataFrame(rows2, columns=["kernel", "test"])
display(result)


In [ ]:
# 完整答案：把 "poly" 加入核函数列表，其余与示例 1 一致。
kernel_list = ["linear", "rbf", "poly"]

rows2 = []
for kernel in kernel_list:
    m = make_pipeline(StandardScaler(), SVC(kernel=kernel, C=1)).fit(
        X_train, y_train
    )
    rows2.append([kernel, round(m.score(X_test, y_test), 3)])
result = pd.DataFrame(rows2, columns=["kernel", "test"])
print(result.to_string(index=False))


## 95.4 示例 2：C 与 gamma

参数共同控制 RBF 边界复杂度。


In [ ]:
grid = []
for c in [0.1, 1, 10]:
    for gamma in ["scale", 0.01, 0.1]:
        m = make_pipeline(StandardScaler(), SVC(C=c, gamma=gamma)).fit(
            X_train, y_train
        )
        grid.append([c, str(gamma), m.score(X_test, y_test)])
display(
    pd.DataFrame(grid, columns=["C", "gamma", "test"])
    .pivot(index="C", columns="gamma", values="test")
    .round(3)
)


## 95.5 建模流程提醒

1. **定义问题**：写清楚样本粒度、预测时点、目标变量和业务代价。
2. **建立基线**：先用均值、规则或 Dummy 模型得到最低可接受结果。
3. **准备数据**：只用预测时点可获得的信息，避免目标泄漏和时间穿越。
4. **训练与验证**：在训练/验证数据上选择方案，测试集只用于最终估计泛化表现。
5. **评价与解释**：同时看总体指标、错误切片和结果边界，不能只报一个分数。


## 95.6 独立迁移练习

在不改变数据切分和指标的前提下，比较基线与一个模型设置。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# TODO: 在此粘贴或改写最接近的示例。
# 记录：我改了什么？预期会发生什么？实际观察到什么？
change_note = "待填写"
expected_change = "待填写"
observed_change = "运行后填写"
print({"修改": change_note, "预期": expected_change, "观察": observed_change})


## 95.7 本章实训：模型与基线比较

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

X = pd.DataFrame(
    {"visits": [1, 2, 3, 4, 5, 6], "discount": [0, 0, 1, 1, 1, 2]}
)
y = np.array([12, 15, 19, 23, 27, 31])
baseline = DummyRegressor(strategy="mean").fit(X, y)
model = LinearRegression().fit(X, y)
print("基线预测：", np.round(baseline.predict(X[:2]), 2))
print("模型预测：", np.round(model.predict(X[:2]), 2))
print("基线MAE：", round(mean_absolute_error(y, baseline.predict(X)), 2))
print("模型MAE：", round(mean_absolute_error(y, model.predict(X)), 2))


### 95.7.1 第一个结果怎么读

复杂模型之前先建立基线。只有在同一数据切分和同一指标下超过基线，模型才值得继续分析。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
X_changed = X.copy()
X_changed["visits"] = X_changed["visits"] + 1
changed_prediction = model.predict(X_changed)
print("原始前2个预测：", np.round(model.predict(X[:2]), 2))
print("访问次数+1后的预测：", np.round(changed_prediction[:2], 2))
print("预测变化：", np.round(changed_prediction[:2] - model.predict(X[:2]), 2))


### 95.7.2 第二个结果怎么读

只把一个特征整体加 1，观察预测变化。这个实验只能说明模型的预测响应，不能直接证明真实世界的因果关系。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 95.8 错误恢复：模型特征泄漏怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

data = pd.DataFrame(
    {
        "visits": [2, 4, 6],
        "duration_after_call": [30, 80, 120],
        "target": [0, 1, 1],
    }
)
forbidden = {"target", "duration_after_call"}
features = [col for col in data.columns if col not in forbidden]
print("禁止使用：", sorted(forbidden))
print("安全特征：", features)
print("原因：特征必须在预测时点已经可获得。")


### 95.8.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

如果一个字段在结果发生之后才产生，它即使与目标高度相关，也不能作为预测特征。先定义预测时点，再列可用字段。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 95.9 易错点提醒

- 不缩放就调 C 和 gamma
- 在大数据上直接运行核 SVM
- 用同一测试集挑选核和参数
- 开启 probability=True 后忽略额外校准成本


## 95.10 练习与作业

1. 用 GridSearchCV 搜索 C=[0.1,1,10]
2. 搜索 gamma=['scale',0.01,0.1]
3. 输出最佳参数

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 95.11 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“用 GridSearchCV 搜索 C=[0.1,1,10]”。
2. **独立完成**：不复制示例代码，完成“搜索 gamma=['scale',0.01,0.1]”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“输出最佳参数”，用一两句话说明你修改了什么。

### 95.11.1 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 95.11.2 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


In [ ]:
from sklearn.model_selection import GridSearchCV

search = GridSearchCV(
    make_pipeline(StandardScaler(), SVC()),
    {"svc__C": [0.1, 1, 10], "svc__gamma": ["scale", 0.01, 0.1]},
    cv=5,
    n_jobs=-1,
)
search.fit(X_train, y_train)
print(search.best_params_, round(search.best_score_, 3))


## 95.12 小结

理解支持向量机的最大间隔思想、核技巧和缩放要求，比较线性核与 RBF 核。


### 95.12.1 你已经掌握

- 训练 SVC 分类器
- 解释 C 与 gamma 的作用
- 使用 StandardScaler
- 比较线性和非线性决策边界


### 95.12.2 验收标准

- 输入、计算和输出单元格完整。
- 关键变量类型、形状或数值可核对。
- 结论引用输出证据，并注明适用范围。


### 95.12.3 需要注意

- 不缩放就调 C 和 gamma
- 在大数据上直接运行核 SVM
- 用同一测试集挑选核和参数
- 开启 probability=True 后忽略额外校准成本


### 95.12.4 完成检查

- [ ] 能够训练 SVC 分类器
- [ ] 能够解释 C 与 gamma 的作用
- [ ] 能够使用 StandardScaler
- [ ] 能够比较线性和非线性决策边界


### 95.12.5 排错顺序

1. 从上到下重新运行依赖单元格。
2. 检查变量类型、列名、形状和缺失值。
3. 缩小输入范围，定位产生错误的最小步骤。
4. 修复后重新运行完整流程。
